### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

#### Pydantic  
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [13]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model(
     "groq:llama-3.3-70b-versatile",
    
    
)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.0', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000221AAA4E490>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000221AAA4EE90>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [7]:
from pydantic import BaseModel, Field

class Movie(BaseModel): # movie is inherited from class-BaseModel that is a base class for creating pydantic models
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of this movie")
    rating:float=Field(description="The movie rating out of 10")
    actor:str=Field(description="top 3 main actors in this movie")

# Whenever I ask the LLM for movie information, the response must contain these fields with these data types

model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.0', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000221A827B770>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000221A8410590>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'pro

In [8]:
model_with_structure.invoke("Provide details of hte movie- inception")
# Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8, actor='Leonardo DiCaprio')

### Message output alongside parsed output

In [11]:
from pydantic import BaseModel, Field

class Movie(BaseModel): # movie is inherited from class-BaseModel that is a base class for creating pydantic models
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of this movie")
    rating:float=Field(description="The movie rating out of 10")

# Whenever I ask the LLM for movie information, the response must contain these fields with these data types

model_with_structure=model.with_structured_output(Movie, include_raw=True) # we can display the raw output by the llm by setting include_raw=True
print(model_with_structure)
model_with_structure.invoke("Provide details of hte movie- inception")

first={
  raw: _ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.0', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000221A827B770>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000221A8410590>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'par

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'fydfvk0q7', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.5,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 280, 'total_tokens': 313, 'completion_time': 0.035861284, 'completion_tokens_details': None, 'prompt_time': 0.033527304, 'prompt_tokens_details': None, 'queue_time': 0.055301105, 'total_time': 0.069388588}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f8a1a-32b6-7e62-a7fc-523a22797ebd-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'rating': 8.5, 'title': 'Inception', 'year': 2010}, 'id': 'fydfvk0q7', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 280, 'output_tokens': 33, 't

### Nested Structure
A nested structure in Pydantic means that one model contains another model as one of its fields. In your example, `MovieDetails` is the parent model, and its `cast` field is defined as `list[Actor]`, meaning each item in the `cast` list must be an `Actor` object with its own `name` and `role` fields. Instead of storing simple strings, `MovieDetails` stores structured `Actor` objects, allowing you to represent hierarchical or real-world data in a clean, organized, and type-safe manner.


In [ ]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

# MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur')], genres=['Action', 'Sci-Fi'], budget=160.0)

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur')], genres=['Action', 'Sci-Fi'], budget=160.0)

### TypedDict
TypeDict provides a simpler alternative using Python's built-in typing, ideal when you dont need runtime validation

### DataClasses
A data class is a class typically containing mainly data, although there arent really any restrictions. You create it using the @dataclass decorator.It's an alternative to pydantic

In [18]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""

    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")


agent = create_agent(
    model,
    response_format=ContactInfo,  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

name='John Doe' email='john@example.com' phone='(555) 123-4567'
